In [17]:
from helper import *

# Load all clean datasets (remember to re-run the clean_data.py code if the raw data files change)
snis_sewer, snis_water, ibnet_sewer, ibnet_water = load_datasets()

# Create cross-sectional versions of snis sewer and water
CS_YEAR = 2020
snis_sewer_cs, snis_water_cs = snis_sewer[snis_sewer['Year'] == CS_YEAR], snis_water[snis_water['Year'] == CS_YEAR]

In [18]:
# List the name of Y and M for each dataset (N is always 'Total Population')
datasets = [snis_sewer_cs, snis_water_cs, ibnet_sewer, ibnet_water]
names = ['SNIS Sewers', 'SNIS Water', 'IBNET Sewers', 'IBNET Water']
y_variables = ['Sewer Length', 'Water Length', 'Sewer Length', 'Water Length']
m_variables = ['Attended Population (sewer)', 'Attended Population (water)', 'Served Population', 'Served Population']

# Create properly named columns for all the models
for d, y, m in zip(datasets, y_variables, m_variables):
    d['Y'] = d[y]
    d['N'] = d['Total Population']
    d['M'] = d[m]
    d['k'] = d['M'] / d['N']

In [19]:
import statsmodels.api as sm
import scipy.stats as stats

# Define the format of the table
table_1 = {
    'Dataset': names, 'Networks': [], 'Mean k_i': [],                            # Information about the dataset
    'Beta N': [], 'R^2 N': [], 'BIC N': [], 'corr N': [], 'corr p N': [],        # Information about the traditional model
    'Beta M': [], 'R^2 M': [], 'BIC M': [], 'corr M': [], 'corr p M': []         # Information about the generalized model
}

# Function to get the correlation between the model error and the served percentage
def get_error_and_k_correlation(input_model, input_variables, input_labels, k_values):
    predictions = input_model.predict(input_variables)
    error = input_labels - predictions
    return np.corrcoef(error, k_values)[0, 1]

# Function to get the correlation between the model error and the served percentage
def get_error_and_k_correlation_with_p(input_model, input_variables, input_labels, k_values):
    predictions = input_model.predict(input_variables)
    error = input_labels - predictions
    return stats.pearsonr(error, k_values)

# Populate the fields
for dataset in datasets:
    table_1['Networks'].append(dataset.shape[0])
    table_1['Mean k_i'].append(round(np.mean(dataset['k']), 2))
    models = []

    for population in ['N', 'M']:

        # Fit the model using statsmodel after adding a constant
        X, y = sm.add_constant(np.log(dataset[[population]])), np.log(dataset['Y'])
        model = sm.OLS(y, X).fit()

        # Add the information from SM to the table
        table_1['Beta ' + population].append(round(model.conf_int(1).loc[population, 0], 2))
        table_1['R^2 ' + population].append(round(model.rsquared, 2))
        table_1['BIC ' + population].append(int(round(model.bic, 0)))
        corr_coefficient, p_value = get_error_and_k_correlation_with_p(model, X, y, dataset['k'])
        table_1['corr ' + population].append(corr_coefficient)
        table_1['corr p ' + population].append(p_value)
        models.append(model)

table_1 = pd.DataFrame(table_1)
table_1['Delta BIC'] = table_1['BIC M'] - table_1['BIC N']
table_1

,Dataset,Networks,Mean k_i,Beta N,R^2 N,BIC N,corr N,corr p N,Beta M,R^2 M,BIC M,corr M,corr p M,Delta BIC
0,SNIS Sewers,2755,0.58,0.78,0.50,7946,0.700627,0.000000e+00,0.84,0.80,5483,0.028451,1.354422e-01,-2463
1,SNIS Water,2689,0.77,0.91,0.78,5028,0.542283,1.831961e-205,0.87,0.85,4040,0.079379,3.775346e-05,-988
2,IBNET Sewers,564,0.61,0.82,0.65,1646,0.537348,1.637850e-43,0.75,0.75,1445,0.079697,5.855516e-02,-201
3,IBNET Water,890,0.77,0.66,0.54,2668,0.570938,4.012571e-78,0.70,0.66,2402,0.288949,1.413307e-18,-266


In [23]:
# Determine the variables used in each model
models = {
    'Traditional Scaling': ['N'],
    'Exploratory Model': ['N', 'k'],
    'Generalized Scaling': ['M'],
}

# Function to format the confidence interval
def get_confidence_interval(input_model, variable, decimals=2):
    try:
        value = f"{input_model.conf_int(1).loc[variable, 0]:.{decimals}f}"
        lower_bound = f"{input_model.conf_int().loc[variable, 0]:.{decimals}f}"
        upper_bound = f"{input_model.conf_int().loc[variable, 1]:.{decimals}f}"
        return f'{value} [{lower_bound}, {upper_bound}]'
    except KeyError:
        return '-'

tables = []

# Prepare Table 2 and alternative versions
for dataset, name in zip(datasets, names):

    # Print the name of the model for us to identify when copying information
    print(name)

    # Template for Table 2, note that it will be transposed
    table_2 = {'Name': [], 'ln($Y_0$)': [], r'$\beta$': [], '$\lambda$': [], 'Adj. R$^2$': [], 'BIC': [], 'F Stat.': [], r'$\rho$($\epsilon$, k)': []}

    for model_name, x_variables in models.items():

        # Fit the model using statsmodel after adding a constant
        X, y = sm.add_constant(np.log(dataset[x_variables])), np.log(dataset['Y'])
        model = sm.OLS(y, X).fit()

        # Add the information from SM to the table
        table_2['Name'].append(model_name)
        table_2['ln($Y_0$)'].append(get_confidence_interval(model, 'const', 2))
        table_2[r'$\beta$'].append(get_confidence_interval(model, x_variables[0], 2))
        table_2['$\lambda$'].append(get_confidence_interval(model, 'k', 2))
        table_2['Adj. R$^2$'].append(round(model.rsquared_adj, 2))
        table_2['BIC'].append(int(round(model.bic, 0)))
        table_2['F Stat.'].append(int(round(model.fvalue, 0)))
        table_2[r'$\rho$($\epsilon$, k)'].append(get_error_and_k_correlation(model, X, y, dataset['k']))

    # Create a dataframe and format it to match the output table
    table_2_output = pd.DataFrame(table_2)
    table_2_output.set_index('Name', inplace=True)
    table_2_output = table_2_output.T

    # Adjust the number of significant figures shown
    table_2_output = table_2_output.applymap(lambda x: f"{x:.2f}" if isinstance(x, float) else x)

    tables.append(table_2_output)

SNIS Sewers
SNIS Water
IBNET Sewers
IBNET Water


In [25]:
# SNIS Sewers
tables[0]

Name,Traditional Scaling,Exploratory Model,Generalized Scaling
ln($Y_0$),"-4.09 [-4.39, -3.80]","-3.94 [-4.12, -3.75]","-3.97 [-4.12, -3.83]"
$\beta$,"0.78 [0.76, 0.81]","0.84 [0.82, 0.86]","0.84 [0.83, 0.86]"
$\lambda$,-,"0.85 [0.82, 0.87]",-
Adj. R$^2$,0.50,0.80,0.80
BIC,7946,5490,5483
F Stat.,2758,5360,10722
"$\rho$($\epsilon$, k)",0.70,0.02,0.03


In [26]:
# SNIS Water
tables[1]

Name,Traditional Scaling,Exploratory Model,Generalized Scaling
ln($Y_0$),"-4.66 [-4.83, -4.48]","-3.86 [-4.02, -3.71]","-4.00 [-4.14, -3.87]"
$\beta$,"0.91 [0.89, 0.93]","0.86 [0.85, 0.88]","0.87 [0.86, 0.89]"
$\lambda$,-,"0.97 [0.91, 1.02]",-
Adj. R$^2$,0.78,0.85,0.85
BIC,5028,4036,4040
F Stat.,9762,7683,15292
"$\rho$($\epsilon$, k)",0.54,0.02,0.08


In [27]:
# IBNET Sewers
tables[2]

Name,Traditional Scaling,Exploratory Model,Generalized Scaling
ln($Y_0$),"-4.37 [-4.93, -3.81]","-3.05 [-3.55, -2.55]","-3.08 [-3.46, -2.71]"
$\beta$,"0.82 [0.77, 0.87]","0.74 [0.70, 0.79]","0.75 [0.71, 0.78]"
$\lambda$,-,"0.75 [0.66, 0.85]",-
Adj. R$^2$,0.64,0.75,0.75
BIC,1646,1452,1445
F Stat.,1023,848,1699
"$\rho$($\epsilon$, k)",0.54,0.07,0.08


In [28]:
# IBNET Water
tables[3]

Name,Traditional Scaling,Exploratory Model,Generalized Scaling
ln($Y_0$),"-2.18 [-2.65, -1.71]","-1.76 [-2.15, -1.37]","-2.35 [-2.72, -1.98]"
$\beta$,"0.66 [0.62, 0.71]","0.66 [0.63, 0.70]","0.70 [0.67, 0.74]"
$\lambda$,-,"1.11 [1.00, 1.22]",-
Adj. R$^2$,0.54,0.68,0.66
BIC,2668,2353,2402
F Stat.,1036,936,1706
"$\rho$($\epsilon$, k)",0.57,0.08,0.29
